# DataMate Hybrid — Colab Pro vLLM Server + Cloudflare Tunnel (Chế độ Remote GPU)

Notebook này dùng GPU của **Google Colab Pro (L4 24GB / A100 40-80GB / G4 RTX PRO 6000 Blackwell 96GB)** để chạy `vLLM` phục vụ model (chọn ở cell 2: `google/gemma-4-12B-it` hoặc `Qwen/Qwen3-14B-FP8`) qua **Cloudflare Quick Tunnel**.

- **Máy cá nhân (Local không cần GPU):** Chạy toàn bộ Hybrid Pipeline (`FastCore` CPU xử lý ~90% câu chuẩn trong vài ms + gọi sang `LLM_BASE_URL` trên Colab Pro cho các câu paraphrase / rule mới, đồng thời cache toàn bộ response vào `cache/llm/`).

### Hướng dẫn sử dụng:
1. Trên Colab Pro: `Runtime -> Change runtime type -> G4 GPU / A100 GPU / L4 GPU`.
2. (Tuỳ chọn) Thêm Colab Secret tên `VLLM_API_KEY` (hoặc để mặc định `datamate-colab-key`).
3. Chạy toàn bộ các cell từ trên xuống dưới. Cell số 5 sẽ in ra dòng `LLM_BASE_URL=https://xxxx.trycloudflare.com/v1` và `LLM_API_KEY=...`.
4. Dán 2 dòng đó vào file `.env` ở máy local rồi chạy:
   ```bash
   .venv/bin/python -m datamate.predict --data-dir <split_dir> --out-dir ./result --system hybrid --workers 8
   ```

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Chọn model rồi cài vLLM (~3-5 phút)
#   "gemma-4-12b": google/gemma-4-12B-it (05/2026, 11.95B, Apache-2.0), reasoning parser gemma4
#   "qwen3-14b"  : Qwen/Qwen3-14B-FP8 (bản cho public PA 0.651), reasoning parser qwen3
#   "qwen3-14b-lora"     : Qwen3-14B-FP8 + adapter LoRA đọc câu hỏi (MyDrive/datamate/lora-parse)
#   "qwen3-14b-bf16-lora": Qwen3-14B (BF16) + adapter — bản đã nộp Docker
#   "qwen3-14b-bf16-lora-lo": Qwen3-14B (BF16) + 3 adapter leave-one-industry-out
#                             (MyDrive/datamate/lora-lo-{pharmacy,building,restaurant}, notebooks/train_lora_leaveout.ipynb)
MODEL_CHOICE = "qwen3-14b"

MODELS = {  # served name -> (HF model, reasoning parser, dùng LoRA?)
    "qwen3-14b": ("Qwen/Qwen3-14B-FP8", "qwen3", False),
    "gemma-4-12b": ("google/gemma-4-12B-it", "gemma4", False),
    "qwen3-14b-lora": ("Qwen/Qwen3-14B-FP8", "qwen3", True),
    "qwen3-14b-bf16-lora": ("Qwen/Qwen3-14B", "qwen3", True),
    "qwen3-14b-bf16-lora-lo": ("Qwen/Qwen3-14B", "qwen3", True),
}
MODEL_ID, REASONING_PARSER, USE_LORA = MODELS[MODEL_CHOICE]
# Base keeps its own served name (LLM cache is keyed by it); the adapter is served as "datamate-parse".
SERVED_NAME = {"qwen3-14b-lora": "qwen3-14b", "qwen3-14b-bf16-lora": "qwen3-14b-bf16",
               "qwen3-14b-bf16-lora-lo": "qwen3-14b-bf16"}.get(MODEL_CHOICE, MODEL_CHOICE)
LORA_NAME, LORA_DIR = "datamate-parse", "/content/drive/MyDrive/datamate/lora-parse"
LORA_MODULES = {LORA_NAME: LORA_DIR}  # served adapter name -> directory
if MODEL_CHOICE == "qwen3-14b-bf16-lora-lo":
    LORA_MODULES = {f"lo-{f}": f"/content/drive/MyDrive/datamate/lora-lo-{f}"
                    for f in ("pharmacy", "building", "restaurant")}
if USE_LORA:
    from google.colab import drive
    drive.mount("/content/drive")
!pip install -q -U vllm "huggingface_hub[hf_transfer]"


In [ ]:
import os
import subprocess

try:
    from google.colab import userdata
    API_KEY = userdata.get("VLLM_API_KEY") or "datamate-colab-key"
except Exception:
    API_KEY = os.environ.get("VLLM_API_KEY", "datamate-colab-key")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
gpu_mb = int(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]
).split()[0])
if gpu_mb >= 40000:
    # G4 (RTX PRO 6000 Blackwell, 96 GB) hoặc A100 (40/80 GB)
    MAX_LEN, EXTRA = "16384", ["--gpu-memory-utilization", "0.92"]
else:
    # L4 (24 GB): ~15 GB FP8 weights + KV cache cho 12k context
    MAX_LEN, EXTRA = "12288", ["--gpu-memory-utilization", "0.95", "--max-num-seqs", "16"]

print(f"GPU VRAM: {gpu_mb} MB | max-model-len: {MAX_LEN}")
cmd = [
    "vllm", "serve", MODEL_ID,
    "--served-model-name", SERVED_NAME,
    "--max-model-len", MAX_LEN,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api-key", API_KEY,
    "--enable-prefix-caching",
    "--reasoning-parser", REASONING_PARSER,
    "--seed", "0",
    *EXTRA,
    *(["--enable-lora", "--lora-modules", *[f"{k}={v}" for k, v in LORA_MODULES.items()],
       "--max-lora-rank", "16", "--max-loras", str(len(LORA_MODULES))] if USE_LORA else []),
]
vllm_log = open("/content/vllm.log", "w")
vllm = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
print("vLLM starting, pid:", vllm.pid)

In [ ]:
# Chờ tải và khởi chạy model (~5-8 phút lần đầu)
import time
import urllib.request

for _ in range(180):
    if vllm.poll() is not None:
        print(open("/content/vllm.log").read()[-4000:])
        raise RuntimeError("vLLM exited unexpectedly, see log above")
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5)
        print("vLLM server is READY!")
        break
    except Exception:
        time.sleep(10)
!tail -n 5 /content/vllm.log

In [ ]:
# Mở Cloudflare Quick Tunnel lấy HTTPS URL public cho máy local
import os
import re

if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared
tunnel_log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
url = None
for _ in range(30):
    time.sleep(2)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found[0]
        break
print("=== COPY VÀO FILE .env Ở MÁY LOCAL ===")
print(f"LLM_BASE_URL={url}/v1")
print(f"LLM_API_KEY={API_KEY}")
print(f"LLM_MODEL={SERVED_NAME}")
if USE_LORA and len(LORA_MODULES) == 1:
    print(f"LLM_PARSE_MODEL={LORA_NAME}")
elif USE_LORA:
    print("# adapters served:", ", ".join(LORA_MODULES), "(LLM_PARSE_MODEL chọn khi chạy eval)")
print("LLM_DISABLE_THINKING=true")

In [ ]:
# CHỈ CHẠY CELL NÀY KHI ĐÃ PREDICT XONG Ở MÁY LOCAL (bỏ dấu # bên dưới để dừng):
# tunnel.terminate()
# vllm.terminate()
# print("Stopped tunnel and vLLM.")